### Préambule

L'exploration et le nettoyage de notre jeu de données sont des étapes essentielles pour assurer la qualité des analyses ultérieures. Cela implique l'identification et la gestion des valeurs manquantes, des doublons et des anomalies. Ces processus permettent de préparer un ensemble de données fiable et pertinent pour des analyses.

**Warning** : Ce cas pratique prend en moyenne 4h à résoudre. Pour vous permettre d'aller potentiellement plus vite si vous le voulez, les réponses à certaines questions vous seront déjà fournies.

Elles seront marquée d'un "Bonus" et les réponses seront cachées dans un toggle à dérouler.

A vous de les consulter, ou non, de choisir le mode facile ou difficile.
Prêt ? C'est parti !

### Contexte

Financia Solutions, est une entreprise de services financiers où Pierre, un comptable expérimenté, se trouve confronté à une nouvelle problématique. Pierre gère une grande quantité de transactions financières et, chaque année, il remarque des variations saisonnières dans les dépenses et les revenus de ses clients. Cependant, il n’a jamais eu le temps ni les outils pour analyser ces tendances en détail.

Pierre décide qu'il est temps de mieux comprendre la saisonnalité des transactions pour anticiper les flux financiers et adapter ses conseils en fonction des périodes d’activité.
Pour l’aider dans cette mission, il sollicite un Data Analyst pour l'aider dans la préparation de ses données.

Pierre fournit deux premiers datasets pour cette analyse :



*   Transactions : Ce dataset inclut des informations sur chaque transaction,comme le montant, le type de transaction, le pays et le département.
*  Dates : Ce dataset comporte des données temporelles qui complètent les transactions, permettant une analyse par mois, jour et heure.





## Etape 1 : Import des packages et jeux de données nécessaires

#### 1. Importer les packages pandas, numpy, seaborn, matplotlib.pyplot ainsi que les jeux de données.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

## Etape 2 : Analyses primaires

#### **2**. Visualiser les deux DataFrames `transaction` et `date`

In [ ]:
df_Country_Region = pd.read_csv('Country_Region.csv')
df_Date = pd.read_csv('date.csv')
df_Transaction = pd.read_csv('transaction.csv')

In [ ]:
df_Transaction

In [ ]:
df_Date

#### 3. Quelles sont les dimensions des datasets (nombre de lignes et colonnes) ?

#### 4. Quelles sont les informations des DataFrames (type des colonnes, valeurs non nulles, etc. ...) ?

La méthode `.info()` en Python fournit un résumé concis des informations sur un DataFrame, y compris le nombre total de lignes, le nombre de valeurs non nulles par colonne, le type de données de chaque colonne et la mémoire utilisée. Cette méthode est utile pour avoir une vue d'ensemble rapide de la structure et de la composition des données, facilitant ainsi le diagnostic des problèmes de qualité des données.

## Etape 3 : Jointures et concaténation

#### *Bonus* 5. Identifier la clé de jointure et la clé primaire. Dans notre cas, notre clé de jointure est également notre clé primaire.

*Indice: plusieurs colonnes semblent être communes et pourraient être une clé de jointure, mais une seule d'entre elles valide les conditions de ce que doit être une clé primaire*

Pour rappel: une clé primaire doit respecter les contraintes d'unicité et de valeurs non nulles. Nous allons donc vérifier si c'est le cas pour la colonne `Unnamed: 0` qui semble à première vue représenter une transaction par ligne

In [ ]:
df_Transaction['Transaction Type'].is_unique

##### Dérouler le toggle pour obtenir la réponse à la question 5

In [ ]:
#Vérifions qu'il n'y ai pas de valeurs manquantes dans la colonne `Unnamed: 0`
print(df_Transaction['Unnamed: 0'].isnull().any())  # False signifie qu’il n’y a pas de valeurs manquantes
print(df_Date['Unnamed: 0'].isnull().any())  # False signifie qu’il n’y a pas de valeurs manquantes

In [ ]:
#Vérifions que chaque valeur dans "Unnamed: 0" est unique dans les deux DataFrames :
print(df_Transaction['Unnamed: 0'].is_unique)  # True signifie qu’il n’y a pas de doublon
print(df_Date['Unnamed: 0'].is_unique)  # True signifie qu’il n’y a pas de doublon

On identifie  les colonnes `Unnamed: 0`, `Account Number`, `Transaction Country`, et `Name` comme étant des colonnes communes puisqu'elles possèdent le même nom.
Néanmoins, seulement `Unnamed: 0` semble pouvoir créer une jointure logique entre nos DataFrames.

En effet:

*   `Unnamed: 0` est dans les deux dataset et semble être un index, probablement utile pour une jointure qui représentera une ligne par transaction effectuée;
* `Name` est dans les deux datasets, et est une colonne qui ne contient que des valeurs NaN et ne peut donc être exploitée;
*   `TransactionCountry` est dans les deux datasets mais ne ressemble pas à une clé unique puisqu'elle contient deux valeurs: `United States of America`et `USA`
* `Account Number` semble être le numéro de compte, mais un numéro de compte peut effectuer plusieurs transactions.

La méthode **`.any()`** en Python, appliquée sur une structure comme une Series ou un DataFrame Pandas, **vérifie si au moins une valeur est vraie ou non nulle** dans l’ensemble analysé.

Sur une Series, elle retourne True si une valeur au moins est vraie.
Sur un DataFrame, .any() peut s'appliquer par colonne (par défaut) ou par ligne, retournant True pour chaque colonne ou ligne contenant au moins une valeur vraie.

In [ ]:
#Bonus: on peut créer une fonction qui vient vérifier nos intuitions sur les autres colonnes qui ont le même nom dans chacun des DataFrames

def check_primary_key_columns(df1, df2, columns):
    results = {}
    for column_name in columns:
        # Vérifier l'absence de valeurs NaN
        if df1[column_name].isnull().any() or df2[column_name].isnull().any():
            results[column_name] = f"La colonne '{column_name}' contient des valeurs NaN."
            continue

        # Vérifier l'unicité des valeurs dans chaque DataFrame
        if not df1[column_name].is_unique or not df2[column_name].is_unique:
            results[column_name] = f"La colonne '{column_name}' contient des doublons."
            continue

        # Si toutes les conditions sont remplies
        results[column_name] = f"La colonne '{column_name}' peut être une clé primaire et de jointure entre les deux DataFrames."

    return results

In [ ]:
# Liste des colonnes à vérifier
columns_to_check = ["Unnamed: 0", "Account Number", "TransactionCountry", "Name"]

#Vérifions pour les deux DataFrames transaction et date

results = check_primary_key_columns(transaction, date, columns_to_check)

# Afficher les résultats
for column, result in results.items():
  print(result)

#### 6. Supprimer les colonnes qui sont identiques dans les deux DataFrames pour éviter les colonnes doublées lors de la jointure.

In [ ]:
df_Date.columns

In [ ]:
df_Date = df_Date.drop(columns=['Account Number','TransactionCountry', 'Name' ])

#### 7. Concaténer les DataFrames sur la colonne qui correspond à la clé primaire et appeler le DataFrame final `management_accounting`


In [ ]:
df_management_accounting = pd.merge(df_Transaction, df_Date, on='Unnamed: 0')
df_management_accounting = df_management_accounting.set_index('Unnamed: 0')
df_management_accounting

In [ ]:
df_management_accounting.rename_axis("Index")

## Etape 4 : Supprimer les doublons

Bien que notre colonne d'index soit unique, il est important de comprendre que dans la réalité, certaines informations peuvent être enregistrées plusieurs fois (à cause de bugs informatiques, d'oublis humains, ...).
Dès lors, on s'expose au problème des doublons.
C'est pour cela que l'on souhaite utiliser notre colonne qui représente les index en tant qu'Index dans un DataFrame pour en suite venir faire une analyse sur les doublons pour s'en débarasser.

*Rappel : un doublon est une ligne qui contient exactement les mêmes valeurs, dans notre cas, une transaction qui aurai été enregistrée deux fois par erreur*

#### *Bonus* 8. Définir la colonne Unnamed: 0 comme index, et la renommer 'Index'

#####Dérouler le toggle pour obtenir la réponse à la question 8

In [ ]:
# Définir 'Unnamed: 0' comme index et la renommer 'Index'
management_accounting.rename(columns={'Unnamed: 0': 'Index'}, inplace=True)
management_accounting.set_index('Index', inplace=True)
# Vérifier le résultat
management_accounting#.head()

#### 9. Vérifier s'il y a des doublons dans le DataFrame, si oui, les supprimer

In [ ]:
## compte le nbr de doublons
df_management_accounting.duplicated().sum()

In [ ]:
## Selection les lignes qui ont des doublons
df_duplicates = df_management_accounting[df_management_accounting.duplicated(keep=False)]

## Trier les doublons pour les regrouper
df_duplicates = df_duplicates.sort_values(by=list(df_duplicates.columns))

## Afficher les doublons
df_duplicates

In [ ]:
## Retire les doublons
df_management_accounting = df_management_accounting.drop_duplicates()

####10. Mettre à jour l'Index pour que les lignes supprimées ne puissent pas impacter l'indexation logique des lignes

In [ ]:
df_management_accounting

In [ ]:
df_management_accounting = df_management_accounting.reset_index()
df_management_accounting = df_management_accounting.drop(columns=['Unnamed: 0'])
df_management_accounting

In [ ]:
df_management_accounting = df_management_accounting.rename_axis("Index")
df_management_accounting

##Etape 5 : Traiter les valeurs manquantes

#### 11. Identifier les colonnes possédants des valeurs manquantes et afficher les valeurs contenues dans chaque colonnes ayant des valeurs manquantes

In [ ]:
## identifie les colonnes avec des nuls"
df_management_accounting.isna().sum()

####12 . Observer les valeurs présentes dans la colonne "Project", et procéder à la transformation adéquate.
*Indice : On souhaite que toutes nos valeurs inconnues soient représentées sous forme de Nan*

In [ ]:
df_management_accounting['Project'].unique()

On remarque qu'il y a des valeurs '???' qui semblent vouloir indiquer que le projet en question n'est pas renseigné, et donc est inconnu, nous allons vouloir le remplacer par une valeur NaN

In [ ]:
df_management_accounting['Project'] = df_management_accounting['Project'].replace('???' , np.nan)

####13. *Bonus* Compter le pourcentage de valeurs manquantes dans chaque colonne et procéder à la gestion des ces dernières selon les règles suivantes.

Si une colonne à moins de 5% de valeurs manquantes, supprimons les lignes ayant des valeurs manquantes. Si elle a plus de 80% de valeurs manquantes, supprimer la colonne.


In [ ]:
df_management_accounting.isna().mean()*100

In [ ]:
temp = df_management_accounting

In [ ]:
for x in temp.columns:
  print(x)
  if 0 <temp[x].isna().mean()*100 < 5:
    temp = temp.dropna(subset=[x])
    print(x, 'ça garde juste les bonnes lignes')
  elif temp[x].isna().mean()*100 > 80:
    temp = temp.drop(columns=[x])
    print(x, 'ça dégage')
  else:
    print(x, 'Pas de changement')
temp.isna()

In [ ]:
temp = temp.reset_index()
temp = temp.drop(columns=['Index'])
temp = temp.rename_axis("Index")
temp

In [ ]:
df_management_accounting = temp
df_management_accounting

In [ ]:
df_management_accounting.isna().sum()

#####Dérouler le toggle pour obtenir la réponse de la question 13

In [ ]:
colonnes_valeurs_manquantes = list(management_accounting.columns[management_accounting.isna().any()]) #On récupère une liste de ces colonnes
# Calculer le pourcentage de valeurs manquantes pour chaque colonne
missing_percent = management_accounting.isna().mean() * 100

# Parcourir chaque colonne avec des valeurs manquantes
for column in colonnes_valeurs_manquantes:
    # Si le pourcentage de valeurs manquantes est inférieur à 5%, supprimer les lignes avec NaN dans cette colonne
    if missing_percent[column] < 5:
        management_accounting = management_accounting.dropna(subset=[column])

    # Si le pourcentage de valeurs manquantes est supérieur à 80%, supprimer la colonne
    elif missing_percent[column] > 80:
        management_accounting = management_accounting.drop(columns=[column])

# Afficher le DataFrame résultant
management_accounting

Il existe une multitude de manière de gérer des valeurs manquantes, on peut :


*   Supprimer les valeurs manquantes (lignes ou colonnes)
*   Remplacer les valeurs manquantes par des valeurs spécifiques
* Interpoler les valeurs manquantes
* Utiliser des modèles plus complexes pour prédire les valeurs manquantes (modèles de ML)
* Marquer les valeurs manquantes (créer une colonne qui indique clairement si il y a une valeur manquante ou non)

Il n'existe aucune règle permettant de définir ce qu'il faut faire dans quel cas, et cela dépend uniquement du cas d'usage. Dès lors, c'est au Data Analyst de bien comprendre les besoins métiers pour faire un choix dans la gestion de ces valeurs.  

## Etape 6 : Traiter les types de données

#### 14. Afficher le type de données de toutes les colonnes

In [ ]:
df_management_accounting.info()
df_management_accounting

####15. Procéder à la résolution des problématiques sur les types de données suivantes.

On identifie deux principales problématiques à résoudre:



* la colonne `Transaction Amount` n'est pas du type `float64`
* les colonnes Date et Time doivent être fusionnées, au format `datetime64` et supprimées par la suite




In [ ]:
df_management_accounting.dtypes

In [ ]:
df_management_accounting['Transaction Amount'].str.replace().astype(float) ## correction

In [ ]:
df_management_accounting['Transaction Amount'] = df_management_accounting['Transaction Amount'].apply(lambda x: x if type(x) == float else float(x.replace(',','.')))
df_management_accounting

In [ ]:
df_Date_Time = df_management_accounting['Date'] + ' ' + df_management_accounting['Time']
df_Date_Time = df_Date_Time.rename('Date_Time')
df_Date_Time

In [ ]:
df_Date_Time = pd.to_datetime(df_Date_Time)
df_Date_Time

In [ ]:
df_management_accounting = pd.concat([df_management_accounting, df_Date_Time], axis=1)
df_management_accounting

In [ ]:
df_management_accounting = df_management_accounting.drop(columns=['Date', 'Time'])
df_management_accounting

## Etape 7 : Traiter les colonnes catégorielles

####16. Sélectionner uniquement les données catégorielles et déterminer quelles sont les valeurs présentes dans chacune de ces colonnes ?

*Indice  : En particulier les colonnes : `Transaction Type` , `Category` , `Department` , `TransactionCountry`, `Project` , `Location` , `Note`*

In [ ]:
for x in df_management_accounting.columns:
  print("Valeur unique de ", x, df_management_accounting[x].unique())
  print('')

####*Bonus* 17. Standardiser les valeurs qui ne le sont pas
*Indice : dans les colonnes `Department` , `TransactionCountry` et `Location`, des valeurs se ressemblent étrangement*

In [ ]:
stand = df_management_accounting['Department']
stand = stand.replace("Analyst Data", "Data Analyst")
stand.unique()

#####Dérouler le toggle pour obtenir la réponse à la question 17

In [ ]:
# Remplacer 'Analyst Data' par 'Data Analyst' dans la colonne 'Department'
df_management_accounting['Department'] = df_management_accounting['Department'].replace('Analyst Data', 'Data Analyst')

# Remplacer 'USA' par 'United States of America' dans la colonne 'TransactionCountry'
df_management_accounting['TransactionCountry'] = df_management_accounting['TransactionCountry'].replace('USA', 'United States of America')

# Remplacer 'New-York' par 'New York' dans la colonne 'Location'
df_management_accounting['Location'] = df_management_accounting['Location'].replace('New-York', 'New York')

df_management_accounting

In [ ]:
#Vérifier que les changements ont été appliqués
for i in colonnes_categorielles:
  print(f"La colonne", i, "contient ces valeurs : ", management_accounting[i].unique())

Encore une fois, le traitement des données est très dépendant du cas business. C'est le rôle du Data Analyst de comprendre que deux valeurs peuvent signifier la même chose, et de vouloir potentiellement les standardiser pour qu'elles soient intérprétées comme tel.

Il y a plein de manière de traiter nos valeurs catégorielles:


*   Remplacer les valeurs pour les standardiser
*   Catégoriser des valeurs (*Remplacer Mexico et USA par America pour faire une analyse par continent par exemple*)
* Ordonner des catégories (*Si vos catégories ont un ordre logique (par exemple, Low, Medium, High), vous pouvez les convertir en un type CategoricalDtype ordonné, ce qui peut aider les modèles à mieux interpréter la relation*)
* ...

Au Data Analyst d'établir ces règles en fonction du cas d'usage.

####18. Rajouter l'information du continent correspondant au Pays
*Indice: utiliser le 3ème fichier csv fourni par Pierre : "Country_Region" pour avoir un code pérenne qui permettrait de rajouter n'importe quelle région pour n'importe quel pays*

In [ ]:
df_Country_Region = pd.read_csv('Country_Region.csv')
df_Country_Region

In [ ]:
s = df_Country_Region
s = s.reset_index()
s = s.rename(columns={'index': 'Continent'})
s

In [ ]:
baba = s[';Country;Region'].str.split(';')

In [ ]:
index = []
Country = []
Region = []
for x in baba:
  print(x)
  print('')
  index = index + [x[0]]
  print(index)
  Country = Country + [x[1]]
  print(Country)
  Region = Region + [x[2]]
  print(Region)
  print('')

df_Country_Region = pd.DataFrame({'Index': index, 'Country': Country, 'Region': Region})
df_Country_Region.set_index('Index', inplace=True)
df_Country_Region

In [ ]:
df_management_accounting

## Etape 8 : Traiter les dates

####19. Créer une colonne `Mois` et une colonne `Heure` qui puisse permettre de faire des analyses de saisonalité

## Etape 9 : Traiter les colonnes numériques : les valeurs aberrantes

####*Bonus* 20. Filtrer les valeurs aberrantes pour pouvoir analyser à la fois les transactions "habituelles" et les transactions "exceptionnelles"

Une valeur est considérée comme aberrante si elle est exceptionnelle, c'est à dire si elle est strictement supérieur à 5000$

*Indice: créer un DataFrame `outliers` qui contient ces valeurs aberrantes, et garder le DataFrame `management_accounting` initial avec uniquement les valeurs habituelles*

#####Dérouler ce toggle pour obtenir la réponse à la question 20

In [ ]:
# Filtrer les transactions avec des montants très élevés (exemple : supérieures à 5000)
outliers = management_accounting[management_accounting['Transaction Amount'] > 5000]
print(len(outliers))

In [ ]:
# Filtrer les transactions avec des montants très élevés (exemple : supérieures à 5000)
outliers = management_accounting[management_accounting['Transaction Amount'] > 5000]
print(len(outliers))

#### Bonus: Réperer les valeurs aberrantes en utilisant la méthode Interquartile

Oublions le seuil de 5000€.

En statistiques, pour repérer des valeurs aberrantes (dans la colonne `Transaction Amount` dans notre cas), on peut utiliser plusieurs techniques.
Une méthode courante est basée sur les quartiles pour identifier les valeurs qui s'écartent fortement de la distribution.

Créer les deux DataFrames `outliers` et `management_accounting` selon ce nouveau seuil.

## Bonus : Analyse

Vous avez maintenant deux DataFrames:


*   `management_accounting` qui contient toutes vos transactions habituelles (inférieures ou égales à 5000$)

*   `outliers` qui contient toutes vos transactions exceptionnelles (supérieures à 5000$)

Vous pouvez donc commencer à analyser vos données



####21. Effectuer une analyse de la fréquence des transactions par mois et heure pour chaque catégorie de montant

#### 22. Comparaison des montants moyens par département pour les transactions habituelles et exceptionnelles
